In [1]:
import os

In [2]:
%pwd

'e:\\Replica\\wine_mlops\\research'

In [3]:
os.chdir('../')

In [5]:
%pwd

'e:\\Replica\\wine_mlops'

In [34]:
# Entity is a return type of a function

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [35]:
from wine_quality.constants import *
from wine_quality.utils.common import read_yaml, create_directories

In [36]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
    params_filepath=PARAMS_FILE_PATH,
    schema_filepath=SCHEMA_FILE_PATH):

        """Configuration manager to manage all configurations."""
    
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([
            self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

In [37]:
import os
import urllib.request as request
import zipfile
from pathlib import Path
from typing import List
from wine_quality import logger
from wine_quality.utils.common import get_size

In [38]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"Downloaded file: {filename} of size: {get_size(filename)}")

        else:
            logger.info(f"File already exists: {self.config.local_data_file} of size: {get_size(self.config.local_data_file)}")

    def extract_zip_file(self):
        """Extracts the zip file.
        zip_file_path: str
        Function to extract the zip file.
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [39]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-04-30 15:40:11,193: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-04-30 15:40:11,194: INFO: common: yaml file: params.yaml loaded successfully]
[2025-04-30 15:40:11,195: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-04-30 15:40:11,197: INFO: common: created directory at: artifacts]
[2025-04-30 15:40:11,198: INFO: common: created directory at: artifacts/data_ingestion]
[2025-04-30 15:40:13,894: INFO: 4256222733: Downloaded file: artifacts\data_ingestion\data.zip of size: ~ 23 KB]
